# 📉 Module 1.3 — Measuring Risk I: Volatility
### *From Theory to Real-World Trading with Python*

---

**Course:** Capital Markets Recap — Level 1: Foundations  
**Prerequisites:** Module 1.1 (Financial Returns), Module 1.2 (Asset Classes)  
**Tools:** `yfinance`, `pandas`, `numpy`, `matplotlib`, `scipy`

---

## What This Module Covers

In your lecture notes, the definition of risk is precise:

> *"Riesgo = qué tan volátil es el retorno"*  (Risk = how volatile the return is)

Volatility is the **single most important risk metric** in finance. It drives:
- Options pricing (Black-Scholes)
- Position sizing (Kelly criterion, risk parity)
- Stop-loss placement
- Portfolio construction (mean-variance optimization)
- Regulatory capital requirements (VaR)

This module covers:
1. **Variance and Standard Deviation** — the foundational definitions
2. **Annualizing volatility** — putting it on a comparable scale
3. **Historical (realized) volatility** — computing it from price data
4. **Rolling volatility** — understanding that risk is not constant
5. **Volatility regimes** — bull vs. bear market vol
6. **The VIX** — the market's implied volatility gauge
7. **Confidence intervals on returns** — what volatility tells us about the range of outcomes

---
## 0. Setup

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

TRADING_DAYS = 252
START = '2015-01-01'
END   = '2024-12-31'

print("Setup complete ✅")

In [ ]:
# ── Download data ─────────────────────────────────────────────────────────────
# We'll use SPY (S&P 500) as the primary example, and download VIX alongside

tickers = ['SPY', 'AAPL', 'TSLA', '^VIX']
raw = yf.download(tickers, start=START, end=END, auto_adjust=True, progress=False)['Close']
raw.columns = ['AAPL', 'SPY', 'TSLA', 'VIX']

# Returns (excluding VIX — it's a level, not a price)
returns = raw[['SPY', 'AAPL', 'TSLA']].pct_change().dropna()

spy_ret  = returns['SPY']
aapl_ret = returns['AAPL']
tsla_ret = returns['TSLA']

print(f"Downloaded {len(returns)} trading days for SPY, AAPL, TSLA, VIX")
print(f"Date range: {returns.index[0].date()} → {returns.index[-1].date()}")

---
## 1. Variance and Standard Deviation — The Definitions

### Theory

Starting from first principles. Given a series of returns $R_1, R_2, \ldots, R_T$:

**Step 1 — Expected return (mean):**
$$
\bar{R} = \frac{1}{T} \sum_{t=1}^{T} R_t
$$

**Step 2 — Variance** (average squared deviation from the mean):
$$
\sigma^2 = \frac{1}{T-1} \sum_{t=1}^{T} (R_t - \bar{R})^2
$$

*(We use $T-1$ in the denominator — Bessel's correction — for unbiased estimation from a sample)*

**Step 3 — Standard deviation** (same units as the return):
$$
\sigma = \sqrt{\sigma^2}
$$

Your notes link this directly to risk:
> *"Volatilidad: $(R(v) = 0.5(45\% - 10\%)^2 + 0.5(-25\% - 10\%)^2)$"*

This is exactly the variance formula applied to two probability scenarios (50% chance of +45%, 50% chance of -25%), with expected return = 10%.

### Variance vs. Standard Deviation in practice
- **Variance** ($\sigma^2$) is used in portfolio math — it's what you minimize in mean-variance optimization (Level 2)
- **Standard deviation** ($\sigma$) is used for everything interpretable — it's in the same units as the return itself

In [ ]:
# ── 1a. Replicate your lecture notes' scenario example ───────────────────────
# Investment with E[R]=10%, two scenarios: +45% (prob=0.5) and -25% (prob=0.5)

probs   = np.array([0.5, 0.5])
returns_scenario = np.array([0.45, -0.25])

E_R  = np.sum(probs * returns_scenario)
Var  = np.sum(probs * (returns_scenario - E_R)**2)
SD   = np.sqrt(Var)

print("Lecture notes example — two-scenario investment:")
print(f"  Scenario 1: +45%  (prob=50%)")
print(f"  Scenario 2: -25%  (prob=50%)")
print(f"  Expected Return E[R] = {E_R*100:.1f}%")
print(f"  Variance  σ²         = {Var:.4f}  ({Var*100:.2f}% squared)")
print(f"  Std Dev   σ          = {SD:.4f}  = {SD*100:.2f}%")
print(f"  (Your notes: R(v)=0.1225, R(SD)=0.35 → σ={0.35*100:.0f}%) ✓")

In [ ]:
# ── 1b. Compute from real data — manual vs pandas ────────────────────────────

# Manual computation
mu   = spy_ret.mean()
var_manual = ((spy_ret - mu)**2).sum() / (len(spy_ret) - 1)
sd_manual  = np.sqrt(var_manual)

# Pandas built-in (should match)
sd_pandas  = spy_ret.std()
var_pandas = spy_ret.var()

print("SPY Daily Return Statistics:")
print(f"  Mean return  : {mu*100:.5f}%")
print(f"  Variance  (manual) : {var_manual:.8f}")
print(f"  Variance  (pandas) : {var_pandas:.8f}  ← should match")
print(f"  Std Dev   (manual) : {sd_manual*100:.5f}%")
print(f"  Std Dev   (pandas) : {sd_pandas*100:.5f}%  ← should match")
print(f"  N observations     : {len(spy_ret)}")

---
## 2. Annualizing Volatility

### Theory

Daily volatility is too granular to be meaningful. We need annual volatility to compare across assets, strategies, and time horizons.

**The square-root-of-time rule** — under the assumption that daily returns are i.i.d. (independent and identically distributed), variance scales linearly with time, so standard deviation scales with the square root:

$$
\sigma_{annual} = \sigma_{daily} \times \sqrt{252}
$$

More generally, to convert from one time period to another:

$$
\sigma_{T} = \sigma_{1} \times \sqrt{T}
$$

Where $T$ is the number of periods.

### Why 252?
US equity markets have approximately **252 trading days per year** (365 calendar days minus weekends and ~9 public holidays). Some practitioners use 260 — the choice matters only marginally at the annual level but creates differences in intraday or short-horizon calculations.

> ⚠️ **The i.i.d. assumption is violated in practice.** Returns exhibit **volatility clustering** — periods of high volatility tend to be followed by more high volatility (this is the GARCH effect). The square-root rule underestimates risk over short horizons during crises. We'll see this visually with rolling volatility.

In [ ]:
# ── 2a. Annualized volatility across assets ───────────────────────────────────
assets = {'SPY': spy_ret, 'AAPL': aapl_ret, 'TSLA': tsla_ret}

print(f"{'Asset':<8} {'Daily σ':>10} {'Monthly σ':>12} {'Annual σ':>10} {'Annualization factor':>22}")
print("-" * 68)
for name, ret in assets.items():
    daily   = ret.std()
    monthly = daily * np.sqrt(21)    # ~21 trading days/month
    annual  = daily * np.sqrt(TRADING_DAYS)
    print(f"{name:<8} {daily*100:>9.4f}%  {monthly*100:>10.4f}%  {annual*100:>8.2f}%  "
          f"× √{TRADING_DAYS} = × {np.sqrt(TRADING_DAYS):.2f}")

print("\n→ TSLA is roughly 3x more volatile than SPY — typical for growth stocks")

In [ ]:
# ── 2b. Visualize the square-root-of-time scaling ────────────────────────────
# Show how 1-day vol scales to multi-day vol

days = np.arange(1, 253)
spy_daily_vol = spy_ret.std()

vol_sqrt = spy_daily_vol * np.sqrt(days) * 100  # square root scaling
vol_linear = spy_daily_vol * days * 100          # linear scaling (WRONG — for comparison)

plt.figure(figsize=(12, 5))
plt.plot(days, vol_sqrt, color='steelblue', linewidth=2.5, label='Correct: σ × √T (sqrt-of-time rule)')
plt.plot(days, vol_linear, color='red', linewidth=1.5, linestyle='--', label='Wrong: σ × T (linear scaling)')
plt.axvline(21, color='gray', linestyle=':', alpha=0.7)
plt.axvline(63, color='gray', linestyle=':', alpha=0.7)
plt.axvline(252, color='gray', linestyle=':', alpha=0.7)
plt.text(21, vol_sqrt.max()*0.85, '1M', ha='center', color='gray', fontsize=9)
plt.text(63, vol_sqrt.max()*0.85, '3M', ha='center', color='gray', fontsize=9)
plt.text(252, vol_sqrt.max()*0.85, '1Y', ha='center', color='gray', fontsize=9)
plt.title('SPY — Volatility Scaling Over Time (Square-Root Rule)', fontweight='bold')
plt.xlabel('Horizon (trading days)')
plt.ylabel('Expected Volatility (%)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 3. Historical (Realized) Volatility

### Theory

**Realized (historical) volatility** is simply the sample standard deviation of log returns over a past window:

$$
\sigma_{realized} = \sqrt{\frac{1}{n-1} \sum_{t=1}^{n} (r_t - \bar{r})^2} \times \sqrt{252}
$$

It answers: *"How much did this asset actually fluctuate over the past N days?"*

Common window choices:
- **10-day vol** — very short-term; used in intraday trading and short-term option pricing
- **21-day vol** — 1-month; common in options markets
- **63-day vol** — 1-quarter; used in institutional risk systems
- **252-day vol** — 1-year; used in annual risk reports

### Your notes' confidence interval formula
Your notes derived confidence intervals from volatility:
> $\lim_{sup} = 18.7\% + 4.23\% \times 3 = 31.89\%$  
> $\lim_{inf} = 18.7\% - 4.23\% \times 3 = 6.01\%$

This is a **±3σ interval** — the return is expected to fall within this range 99.7% of the time (under normality). The 4.23% was the **standard error** of the mean return, calculated as $\sigma / \sqrt{n}$.

In [ ]:
# ── 3a. Historical volatility with multiple windows ───────────────────────────
log_ret_spy = np.log(raw['SPY'] / raw['SPY'].shift(1)).dropna()

windows = {'10-day': 10, '21-day': 21, '63-day': 63, '252-day': 252}
vol_df = pd.DataFrame(index=log_ret_spy.index)

for name, w in windows.items():
    vol_df[name] = log_ret_spy.rolling(w).std() * np.sqrt(TRADING_DAYS) * 100

fig, ax = plt.subplots(figsize=(14, 6))
colors_vol = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
alphas     = [0.7, 0.7, 0.9, 1.0]
lws        = [1.0, 1.2, 1.8, 2.5]

for (name, _), color, alpha, lw in zip(windows.items(), colors_vol, alphas, lws):
    ax.plot(vol_df.index, vol_df[name], label=f'HV {name}',
            color=color, linewidth=lw, alpha=alpha)

# Highlight major vol spikes
covid_date = pd.to_datetime('2020-03-16')
idx = vol_df.index.searchsorted(covid_date)
if idx < len(vol_df):
    ax.axvline(vol_df.index[idx], color='black', linestyle='--', alpha=0.5)
    ax.text(vol_df.index[idx], vol_df['10-day'].max() * 0.9, ' COVID\n crash',
            fontsize=9, color='black')

ax.set_title('SPY — Historical Realized Volatility (Multiple Windows, Annualized)',
             fontweight='bold')
ax.set_ylabel('Annualized Volatility (%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\n📊 Key observations:")
print("  - Short windows (10-day) are very spiky and reactive")
print("  - Longer windows (252-day) are smooth but lag — slow to respond to new regimes")
print("  - COVID 2020 caused the largest vol spike in recent history")

---
## 4. Rolling Volatility — Risk Is Not Constant

### Theory

One of the most important empirical facts in finance:

> **Volatility is not constant over time.** It clusters — high-vol periods are followed by high vol, and calm periods are followed by calm. This is called **volatility clustering** or **GARCH effects** (Generalized AutoRegressive Conditional Heteroskedasticity).

This has massive practical implications:
- A single annual volatility estimate misrepresents current risk
- Options become cheaper in calm markets and expensive during crises
- Position sizes must be adjusted dynamically as vol changes
- Many risk management frameworks (VaR, CVaR) use **rolling** vol estimates

**EWMA (Exponentially Weighted Moving Average)** is a popular improvement over simple rolling vol — it gives more weight to recent observations:

$$
\sigma_t^2 = (1 - \lambda) \cdot r_{t-1}^2 + \lambda \cdot \sigma_{t-1}^2
$$

The RiskMetrics standard uses $\lambda = 0.94$ for daily data.

In [ ]:
# ── 4a. Rolling vol vs EWMA vol ───────────────────────────────────────────────

def ewma_volatility(returns, lam=0.94):
    """Compute EWMA volatility (RiskMetrics standard, λ=0.94)."""
    squared = returns ** 2
    ewma_var = squared.ewm(span=(2/(1-lam) - 1), adjust=False).mean()
    return np.sqrt(ewma_var * TRADING_DAYS) * 100

rolling_21  = log_ret_spy.rolling(21).std() * np.sqrt(TRADING_DAYS) * 100
rolling_63  = log_ret_spy.rolling(63).std() * np.sqrt(TRADING_DAYS) * 100
ewma_vol    = ewma_volatility(log_ret_spy)

plt.figure(figsize=(13, 5))
plt.plot(rolling_21.index, rolling_21,  color='#e74c3c', linewidth=1.2, alpha=0.7, label='21-day rolling')
plt.plot(rolling_63.index, rolling_63,  color='#3498db', linewidth=1.8, alpha=0.8, label='63-day rolling')
plt.plot(ewma_vol.index,   ewma_vol,    color='#2c3e50', linewidth=2.0, label='EWMA (λ=0.94)')

# Shade major vol regimes
crisis_periods = [
    ('2018-10-01', '2018-12-31', 'Q4 2018\nSelloff'),
    ('2020-02-15', '2020-05-01', 'COVID'),
    ('2022-01-01', '2022-12-31', '2022\nBear'),
]
for start_c, end_c, label_c in crisis_periods:
    plt.axvspan(pd.to_datetime(start_c), pd.to_datetime(end_c),
                alpha=0.08, color='red')
    mid = pd.to_datetime(start_c) + (pd.to_datetime(end_c) - pd.to_datetime(start_c))/2
    plt.text(mid, rolling_21.max() * 0.85, label_c,
             ha='center', fontsize=8, color='darkred')

plt.title('SPY — Rolling Volatility vs EWMA Volatility (Annualized)',
          fontweight='bold')
plt.ylabel('Annualized Volatility (%)')
plt.legend()
plt.tight_layout()
plt.show()

print("\n📊 EWMA advantages over simple rolling:")
print("  - Reacts faster to rising volatility (more weight on recent data)")
print("  - Falls off more gradually — avoids the 'cliff edge' of rolling windows")
print("  - No need to choose a lookback window — just tune λ")

---
## 5. The VIX — The Market's Fear Gauge

### Theory

The **VIX** (CBOE Volatility Index) is the market's consensus estimate of the S&P 500's **implied volatility** for the next 30 days, expressed as an annualized percentage.

Unlike historical volatility (which looks backward), the VIX is **forward-looking** — it's extracted from current S&P 500 options prices. Investors buying options bid up the price when they expect turbulence, which pushes the VIX up.

**VIX interpretation:**
- **VIX < 15:** Low fear, calm market — typically bull market conditions
- **VIX 15–25:** Normal / moderate uncertainty
- **VIX 25–35:** Elevated anxiety, increased caution
- **VIX > 35:** Fear / panic — often corresponds to sharp market corrections
- **VIX > 80:** Extreme panic (reached ~82 in March 2020, ~80 in October 2008)

### VIX as a contrarian signal
The VIX is **mean-reverting** — extreme spikes tend to be followed by normalization. This creates a contrarian trading signal: when VIX spikes sharply above 40, it has historically been a good time to *buy* equities (not sell), because panic-driven selling often overshoots fair value.

> 💡 The **inverse relationship** between the VIX and the S&P 500 is one of the most consistent empirical regularities in markets: when stocks fall, VIX rises, and vice versa. The average correlation is around -0.70.

In [ ]:
# ── 5a. VIX vs S&P 500 ───────────────────────────────────────────────────────
vix = raw['VIX'].dropna()
spy = raw['SPY'].dropna()

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# S&P 500 price
axes[0].plot(spy.index, spy, color='steelblue', linewidth=1.5)
axes[0].set_title('S&P 500 (SPY)', fontweight='bold')
axes[0].set_ylabel('Price (USD)')

# VIX with color zones
axes[1].plot(vix.index, vix, color='#c0392b', linewidth=1.5)
axes[1].fill_between(vix.index, vix, 0, alpha=0.2, color='#c0392b')

# Horizontal reference lines
for level, label, color in [(15, 'Calm (<15)', '#27ae60'),
                             (25, 'Elevated (25)', '#f39c12'),
                             (35, 'Fear (35)', '#e74c3c')]:
    axes[1].axhline(level, color=color, linestyle='--', linewidth=1.2, alpha=0.8)
    axes[1].text(vix.index[-1], level + 0.5, label, fontsize=8, color=color, ha='right')

axes[1].set_title('VIX — The Fear Index', fontweight='bold')
axes[1].set_ylabel('VIX Level')
axes[1].set_xlabel('Date')

plt.suptitle('S&P 500 vs VIX — When Stocks Fall, Fear Rises', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. VIX vs Realized Volatility — implied vs. actual ─────────────────────
# The VIX is a PREDICTION of future vol; realized vol is what ACTUALLY happened
# Comparing them reveals whether the market over- or under-estimated risk

common_idx = vix.index.intersection(rolling_21.index)
vix_aligned = vix.loc[common_idx]
rvol_aligned = rolling_21.loc[common_idx]

# Volatility risk premium = VIX - Realized Vol (usually positive — fear premium)
vol_premium = vix_aligned - rvol_aligned

plt.figure(figsize=(13, 5))
plt.plot(vix_aligned.index, vix_aligned,   color='#c0392b', linewidth=1.5, label='VIX (implied vol)')
plt.plot(rvol_aligned.index, rvol_aligned, color='#3498db', linewidth=1.5, label='21-day realized vol')
plt.fill_between(vix_aligned.index, vix_aligned, rvol_aligned,
                 where=(vix_aligned >= rvol_aligned), alpha=0.15, color='red',
                 label='Volatility risk premium (VIX > RVol)')
plt.fill_between(vix_aligned.index, vix_aligned, rvol_aligned,
                 where=(vix_aligned < rvol_aligned), alpha=0.25, color='blue',
                 label='VIX < Realized (rare — fear lagged reality)')
plt.title('VIX vs 21-Day Realized Volatility (Annualized %)', fontweight='bold')
plt.ylabel('Volatility (%)')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

avg_premium = vol_premium.mean()
pct_positive = (vol_premium > 0).mean() * 100
print(f"\nVolatility Risk Premium stats ({START[:4]}–{END[:4]}):")
print(f"  Average VIX - Realized Vol : {avg_premium:.2f}%")
print(f"  % of days VIX > Realized  : {pct_positive:.1f}%")
print("\n→ The VIX systematically OVERPREDICTS realized vol — the 'vol risk premium'")
print("  This is why selling options (collecting the premium) is a popular strategy.")

---
## 6. Confidence Intervals on Returns

### Theory

Once we have volatility, we can build **confidence intervals** around expected returns. This directly replicates the formula from your lecture notes.

Under the normality assumption:
$$
P\left(\bar{R} - z_{\alpha/2} \cdot SE \leq R_{true} \leq \bar{R} + z_{\alpha/2} \cdot SE\right) = 1 - \alpha
$$

Where the **standard error** of the mean is:
$$
SE = \frac{\sigma}{\sqrt{n}}
$$

And the critical values are:
- $z_{0.025} = 1.96$ → **95% confidence interval** (±1.96σ)
- $z_{0.005} = 2.58$ → **99% confidence interval** (±2.58σ)
- $z_{0.0015} = 3.00$ → **99.7% confidence interval** (±3σ)

This is **your notes' formula** — the ±3σ interval around 18.7% with SE=4.23%.

### Trading application: Value at Risk (VaR)
**Value at Risk** answers: *"What is the maximum loss I should expect with 95% (or 99%) probability over a given horizon?"*

$$
\text{VaR}_{95\%,\ 1\text{day}} = \bar{R} - 1.645 \cdot \sigma_{daily}
$$

Expressed as a loss on a \$10,000 position:
$$
\text{Dollar VaR} = |\text{VaR}_{\%}| \times \$10,000
$$

In [ ]:
# ── 6a. Replicate your notes' confidence interval example ───────────────────
# Notes: small stocks return = 18.7%, σ = 4.23% (standard error of mean), n=86

mean_r  = 0.187
se      = 0.0423  # standard error (σ/√n)

ci_99   = (mean_r - 3 * se, mean_r + 3 * se)
ci_95   = (mean_r - 1.96 * se, mean_r + 1.96 * se)

print("Lecture notes example — Small Stocks confidence intervals:")
print(f"  Mean return = {mean_r*100:.1f}%, SE = {se*100:.2f}%")
print(f"  99.7% interval (±3σ): [{ci_99[0]*100:.2f}%, {ci_99[1]*100:.2f}%]")
print(f"  → Your notes: [{6.01:.2f}%, {31.89:.2f}%] ✓")
print(f"  95% interval  (±1.96σ): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")

In [ ]:
# ── 6b. Parametric VaR for SPY, AAPL, TSLA ──────────────────────────────────
POSITION = 10_000  # $10,000 position

print(f"1-Day Value at Risk (VaR) — ${POSITION:,} position")
print(f"{'='*65}")
print(f"{'Asset':<8} {'Mean %':>9} {'Daily σ %':>10} {'VaR 95% $':>12} {'VaR 99% $':>12}")
print("-" * 65)

for name, ret in [('SPY', spy_ret), ('AAPL', aapl_ret), ('TSLA', tsla_ret)]:
    mu_d  = ret.mean()
    sig_d = ret.std()
    var_95 = -(mu_d - 1.645 * sig_d) * POSITION
    var_99 = -(mu_d - 2.326 * sig_d) * POSITION
    print(f"{name:<8} {mu_d*100:>8.4f}%  {sig_d*100:>9.4f}%  "
          f"${var_95:>10.2f}   ${var_99:>10.2f}")

print(f"{'='*65}")
print("\nVaR 95%: On any given day, you lose MORE than this amount only 5% of the time")
print("VaR 99%: On any given day, you lose MORE than this amount only 1% of the time")

In [ ]:
# ── 6c. Visualize return distribution with VaR cutoffs ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

asset_data = [('SPY', spy_ret, '#2980b9'),
              ('AAPL', aapl_ret, '#27ae60'),
              ('TSLA', tsla_ret, '#c0392b')]

for ax, (name, ret, color) in zip(axes, asset_data):
    mu_d, sig_d = ret.mean(), ret.std()

    # Histogram
    ax.hist(ret * 100, bins=80, density=True, color=color, alpha=0.5, label='Actual')

    # Normal fit
    x = np.linspace(ret.min(), ret.max(), 300)
    ax.plot(x * 100, stats.norm.pdf(x, mu_d, sig_d) / 100,
            color='black', linewidth=1.8, label='Normal fit')

    # VaR lines
    var95 = mu_d - 1.645 * sig_d
    var99 = mu_d - 2.326 * sig_d
    ax.axvline(var95 * 100, color='orange', linewidth=2, linestyle='--',
               label=f'VaR 95%: {var95*100:.2f}%')
    ax.axvline(var99 * 100, color='red', linewidth=2, linestyle='--',
               label=f'VaR 99%: {var99*100:.2f}%')

    ax.set_title(f'{name} Return Distribution', fontweight='bold', fontsize=10)
    ax.set_xlabel('Daily Return (%)')
    ax.set_ylabel('Density' if ax == axes[0] else '')
    ax.legend(fontsize=7)

plt.suptitle('Daily Return Distributions with VaR Thresholds', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print("\n⚠️  Note the fat left tails in all three assets — actual extreme losses")
print("   exceed what the normal distribution predicts. This means parametric VaR")
print("   UNDERESTIMATES tail risk. Historical VaR (using actual return quantiles)")
print("   is more robust for risk management.")

In [ ]:
# ── 6d. Historical VaR vs Parametric VaR — which is more accurate? ──────────
print("Parametric VaR vs Historical VaR comparison:")
print(f"{'='*70}")
print(f"{'Asset':<8} {'Par. VaR 99%':>14} {'Hist. VaR 99%':>15} {'Worst day ever':>16}")
print("-" * 70)

for name, ret in [('SPY', spy_ret), ('AAPL', aapl_ret), ('TSLA', tsla_ret)]:
    par_var99  = (ret.mean() - 2.326 * ret.std()) * 100
    hist_var99 = ret.quantile(0.01) * 100  # 1st percentile of actual returns
    worst_day  = ret.min() * 100
    print(f"{name:<8} {par_var99:>13.2f}%  {hist_var99:>14.2f}%  {worst_day:>15.2f}%")

print(f"{'='*70}")
print("\nHistorical VaR = 1st percentile of ACTUAL observed returns (no distribution assumption)")
print("Worst day = the true tail — normal distribution can't predict this well")

---
## 7. Module Summary & Key Takeaways

| Concept | Formula | Trading Application |
|---|---|---|
| Variance | $\sigma^2 = \frac{1}{T-1}\sum(R_t - \bar{R})^2$ | Input to portfolio optimization |
| Std deviation | $\sigma = \sqrt{\sigma^2}$ | Core risk measure; same units as return |
| Annualized vol | $\sigma_{ann} = \sigma_{daily} \times \sqrt{252}$ | Compare across assets and strategies |
| Historical vol | Rolling window std dev | Current regime risk estimate |
| EWMA vol | $\sigma_t^2 = (1-\lambda)r_{t-1}^2 + \lambda \sigma_{t-1}^2$ | Responsive real-time risk estimate |
| VIX | CBOE S&P 500 implied 30-day vol | Market fear gauge; contrarian signal |
| Parametric VaR | $\bar{R} - z \cdot \sigma$ | Daily loss threshold with given confidence |
| Historical VaR | $R_{\alpha}$ quantile | Fat-tail robust daily loss estimate |

### 🔑 Three Rules to Carry Into Your Trading

1. **Always annualize volatility.** A daily vol of 1.2% sounds small, but it's ~19% annually — that's significant equity-level risk. Always scale to annual for comparability.

2. **Volatility is not constant — respect the regime.** In calm markets (VIX <15), strategies that depend on range-bound behavior work well. In high-vol regimes (VIX >30), mean-reversion breaks down and trend-following tends to outperform.

3. **Use historical VaR over parametric VaR in practice.** Real returns have fat tails. The 1% of the actual return distribution is almost always worse than the normal distribution predicts. Use the actual data.

---

### ➡️ Next: Module 1.4 — Measuring Risk II: Scenarios & Distributions
We'll go deeper into return distributions — exploring skewness, kurtosis, and fat tails. We'll also examine expected shortfall (CVaR), which captures *how bad* losses are beyond the VaR threshold.